# Testes de k na SVD

Imagens selecionadas aleatoriamente, metade Cat e metade Dog. Cada imagem é recortada no maior quadrado central, redimensionada para 300 × 300 pixels com o padrão do Pillow e fatorada em escala de cinza ou RGB conforme escala_cinza. No modo RGB, cada canal recebe sua própria SVD. O MSE considera todos os elementos da imagem preparada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.sparse.linalg import svds
from pathlib import Path

diretorio = Path("dataset/PetImages")

# Configure estes três valores diretamente no notebook.
N = 500
k_max = None
passo = None
escala_cinza = True  # False: manter RGB e fatorar os três canais

N_PIORES = 100
if not isinstance(N, int) or N < 1 or N % 2 != 0:
    raise ValueError("N deve ser um inteiro positivo e par para dividir igualmente entre Cat e Dog.")
if N < N_PIORES:
    raise ValueError(f"N deve ser pelo menos {N_PIORES} para selecionar as {N_PIORES} piores imagens.")
n_gatos = N // 2
n_cachorros = N // 2

if not isinstance(k_max, int) or not isinstance(passo, int):
    raise ValueError("Defina k_max e passo como inteiros na configuração acima.")
if k_max < 1 or passo < 1:
    raise ValueError("k máximo e passo devem ser positivos.")
if k_max >= 300:
    raise ValueError("Com imagens 300 x 300, k_max deve ser menor que 300.")
if passo > k_max:
    raise ValueError("O passo não pode ser maior que k máximo.")
valores_k = list(range(passo, k_max + 1, passo))
print(f"Selecionando {n_gatos} gatos e {n_cachorros} cachorros.")
print("Valores de k:", valores_k)

In [ ]:
rng = np.random.default_rng()
imagens_selecionadas = []
for classe, quantidade in [("Cat", n_gatos), ("Dog", n_cachorros)]:
    pasta = diretorio / classe
    arquivos = sorted(
        arquivo for arquivo in pasta.iterdir()
        if arquivo.is_file() and arquivo.suffix.lower() in [".jpg", ".png"]
    )
    if quantidade > len(arquivos):
        raise ValueError(f"Existem apenas {len(arquivos)} imagens em {classe}, mas foram solicitadas {quantidade}.")
    indices = rng.choice(len(arquivos), size=quantidade, replace=False)
    for indice in indices:
        imagens_selecionadas.append((classe, arquivos[indice]))
print("Total de imagens selecionadas:", len(imagens_selecionadas))

In [ ]:
TAMANHO_ALVO = (300, 300)  # largura, altura; recorte quadrado

def preparar_imagem(A, escala_cinza=True):
    """Recorta o centro em um quadrado e redimensiona para 300 x 300 pixels."""
    if not isinstance(escala_cinza, (bool, np.bool_)):
        raise TypeError("escala_cinza deve ser True ou False.")
    if isinstance(A, Image.Image):
        imagem = A
    else:
        matriz = np.asarray(A)
        if matriz.ndim not in (2, 3) or (matriz.ndim == 3 and matriz.shape[2] != 3):
            raise ValueError("A deve ter formato (altura, largura) ou (altura, largura, 3).")
        if matriz.ndim == 3 and matriz.dtype != np.uint8:
            raise TypeError("Uma matriz RGB deve conter pixels uint8.")
        imagem = Image.fromarray(matriz)
    imagem = imagem.convert("L" if escala_cinza else "RGB")
    largura, altura = imagem.size
    lado_recorte = min(largura, altura)
    esquerda = (largura - lado_recorte) // 2
    topo = (altura - lado_recorte) // 2
    imagem = imagem.crop((esquerda, topo, esquerda + lado_recorte, topo + lado_recorte))
    if imagem.size != TAMANHO_ALVO:
        imagem = imagem.resize(TAMANHO_ALVO)
    return np.asarray(imagem, dtype=float)

def fatorar_canal(canal, k):
    canal = np.asarray(canal, dtype=float)
    if canal.ndim != 2:
        raise ValueError("Cada canal deve ser uma matriz bidimensional.")
    if not isinstance(k, (int, np.integer)) or not 1 <= k < min(canal.shape):
        raise ValueError(f"k deve ser inteiro entre 1 e {min(canal.shape) - 1}.")
    U_k, sigma_k, V_k_T = svds(canal, k=k, which="LM")
    ordem_decrescente = np.argsort(sigma_k)[::-1]
    U_k = U_k[:, ordem_decrescente]
    sigma_k = sigma_k[ordem_decrescente]
    V_k_T = V_k_T[ordem_decrescente, :]
    B_k = U_k
    C_k = sigma_k[:, None] * V_k_T
    return B_k, C_k


def fatorar_imagem(A, k_max):
    if A.ndim == 2:
        return fatorar_canal(A, k_max)
    B_R, C_R = fatorar_canal(A[:, :, 0], k_max)
    B_G, C_G = fatorar_canal(A[:, :, 1], k_max)
    B_B, C_B = fatorar_canal(A[:, :, 2], k_max)
    return B_R, C_R, B_G, C_G, B_B, C_B

def reconstruir(fatores, k):
    if len(fatores) == 2:
        B_k, C_k = fatores
        return B_k[:, :k] @ C_k[:k, :]
    B_R, C_R, B_G, C_G, B_B, C_B = fatores
    canais = [
        B_R[:, :k] @ C_R[:k, :],
        B_G[:, :k] @ C_G[:k, :],
        B_B[:, :k] @ C_B[:k, :],
    ]
    return np.stack(canais, axis=-1)

def calcular_mse(A, A_k):
    return np.mean((A - A_k) ** 2)

def mostrar_imagem(ax, A, titulo):
    if A.ndim == 2:
        ax.imshow(A, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(np.clip(A, 0, 255).astype(np.uint8))
    ax.set_title(titulo)
    ax.axis("off")

In [ ]:
resultados_mse = []
identificadores = []
diretorio_saida = Path("fatores_svd") / ("cinza_300x300" if escala_cinza else "rgb_300x300")
diretorio_saida.mkdir(parents=True, exist_ok=True)
for indice, (classe, caminho) in enumerate(imagens_selecionadas):
    with Image.open(caminho) as imagem:
        A = preparar_imagem(imagem, escala_cinza=escala_cinza)
    fatores = fatorar_imagem(A, k_max)
    pasta_saida = diretorio_saida / classe
    pasta_saida.mkdir(parents=True, exist_ok=True)
    nomes = ("B_k", "C_k") if escala_cinza else ("B_R", "C_R", "B_G", "C_G", "B_B", "C_B")
    np.savez_compressed(pasta_saida / f"{caminho.name}.npz", **dict(zip(nomes, fatores)))
    erros = []
    for k in valores_k:
        A_k = np.clip(reconstruir(fatores, k), 0, 255)
        erros.append(calcular_mse(A, A_k))
    resultados_mse.append(erros)
    identificadores.append((classe, caminho.name))
    if (indice + 1) % 25 == 0:
        print(f"Processadas {indice + 1}/{N} imagens")
resultados_mse = np.array(resultados_mse)
print("Dimensão da matriz de MSE:", resultados_mse.shape)

In [ ]:
mse_medio = np.mean(resultados_mse, axis=0)
plt.figure(figsize=(11, 6))
plt.plot(valores_k, mse_medio, marker="o", linewidth=2)
plt.title("MSE médio das 500 imagens")
plt.xlabel("k")
plt.ylabel("MSE médio")
plt.xticks(valores_k)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mse_por_imagem = np.mean(resultados_mse, axis=1)
indices_ordenados = np.argsort(mse_por_imagem)[::-1]
indices_piores = indices_ordenados[:N_PIORES]
mse_medio_piores = np.mean(resultados_mse[indices_piores, :], axis=0)
plt.figure(figsize=(11, 6))
plt.plot(valores_k, mse_medio_piores, marker="o", linewidth=2)
plt.title("MSE médio das 100 imagens com maiores erros médios")
plt.xlabel("k")
plt.ylabel("MSE médio")
plt.xticks(valores_k)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
N_PIORES_VISUALIZAR = 10
indices_10_piores = indices_ordenados[:N_PIORES_VISUALIZAR]
for posicao, indice in enumerate(indices_10_piores):
    classe, caminho = imagens_selecionadas[indice]
    with Image.open(caminho) as imagem:
        A = preparar_imagem(imagem, escala_cinza=escala_cinza)
    arquivo_svd = diretorio_saida / classe / f"{caminho.name}.npz"
    nomes = ("B_k", "C_k") if escala_cinza else ("B_R", "C_R", "B_G", "C_G", "B_B", "C_B")
    with np.load(arquivo_svd) as dados:
        fatores = tuple(dados[nome] for nome in nomes)
    erros = resultados_mse[indice]
    fig, axes = plt.subplots(1, len(valores_k) + 1, figsize=(3.5 * (len(valores_k) + 1), 4), squeeze=False)
    axes = axes.flatten()
    fig.suptitle(f"{posicao + 1}ª pior imagem — {classe}/{caminho.name}\nMSE médio = {mse_por_imagem[indice]:.2f}", fontsize=16)
    mostrar_imagem(axes[0], A, "Original")
    for j, k in enumerate(valores_k):
        A_k = np.clip(reconstruir(fatores, k), 0, 255)
        mostrar_imagem(axes[j + 1], A_k, f"k = {k}\nMSE = {erros[j]:.2f}")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()
fig.suptitle("Evolução do MSE das 10 piores imagens", fontsize=20)
for posicao, indice in enumerate(indices_10_piores):
    classe, nome = identificadores[indice]
    erros = resultados_mse[indice]
    ax = axes[posicao]
    ax.plot(valores_k, erros, marker="o", linewidth=2)
    ax.set_title(f"{posicao + 1}ª pior — {classe}/{nome}", fontsize=12)
    ax.set_xlabel("k")
    ax.set_ylabel("MSE")
    ax.set_xticks(valores_k)
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
n_valores = []
m_valores = []
for classe, caminho in imagens_selecionadas:
    with Image.open(caminho) as imagem:
        m, n = imagem.size
    n_valores.append(n)
    m_valores.append(m)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(n_valores, edgecolor="black")
axes[0].set_title("Distribuição de n (altura)")
axes[0].set_xlabel("n (pixels)")
axes[0].set_ylabel("Frequência")
axes[0].grid(axis="y", alpha=0.3)
axes[1].hist(m_valores, edgecolor="black")
axes[1].set_title("Distribuição de m (largura)")
axes[1].set_xlabel("m (pixels)")
axes[1].set_ylabel("Frequência")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Proporções das imagens

A proporção de cada imagem é `largura / altura = m / n`. O histograma usa as mesmas imagens selecionadas nas células anteriores; a altura de cada barra mostra quantas imagens têm proporção naquele intervalo.


In [ ]:
proporcoes = np.asarray(m_valores, dtype=float) / np.asarray(n_valores, dtype=float)

plt.figure(figsize=(10, 5))
plt.hist(proporcoes, edgecolor="black")
plt.title("Distribuição das proporções das imagens selecionadas")
plt.xlabel("Proporção (largura / altura)")
plt.ylabel("Número de imagens")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()